# §35 v4 — Zincirleme: kapasite × yazım kuralı

**Otorite ön-kayıt `RESULTS.md` §35'tedir.** Bu markdown kolaylık kopyasıdır.

**Merkezi soru:** state neden **bir** sınırı geçiyor da **ikincisini** geçemiyor?

---

## v3 neden geçersizdi (2026-08-01, koşuldu)

v3 koşuldu ve **geçersiz koşu** olarak kaydedildi — negatif sonuç değil, geçersiz:

1. **Araç eşiğe ulaşmadı.** Taban kol K=0'da eşleşmiş sonda %8.5 (seed 6/8/12/8),
   şans %3.3. §26b aynı sondada **%33.2** almıştı. Bir sınırı geçemeyen modelde
   "neden ikinciyi geçemiyor" sorusu sorulamaz.
2. **Birincil metrik kirliydi.** `carry_example`'ın B chunk etiketleri hem
   chunk-içi P çiftini hem cross-chunk hedefini içeriyor → P=4'te denetlenen 5
   token, **4'ü chunk-içi**. Gözlenen 2.2974, taşıma tamamen şansta (3.40) ve
   chunk-içiler 2.02'de olsa çıkacak değerin **aynısı**: (4×2.02+3.40)/5 = 2.30.

Sebep [Certain]: v2 kapsam kısıntısı **CPU ölçümüne** göre yapıldı (~25 saat),
sonra GPU'ya geçildi ve bir daha ölçülmedi. v3'ün tamamı T4'te **833 sn** sürdü —
12 saatlik limitin %1.9'u. Kısıntı gereksizdi ve aracı bozdu.

## v4'te ne değişti

| | v3 | v4 |
|---|---|---|
| ayarlar | CTX 128, 600 adım, BS 4, CARRY_MAX 8, P 4 | **§26b ayarları:** CTX 256, 1200 adım, BS 8, CARRY_MAX 16, P 6 |
| seed | 4 | **6** |
| birincil metrik | karışık val_loss | **`val_cross`** — yalnız cross-chunk hedef tokenı |
| teşhis | yok | **`val_inchunk`** — chunk-içi çiftler (araç öğrendi mi?) |
| araç kapısı | yok | **taban K=0 ≥ %25 değilse hüküm YOK, koşu geçersiz** |

n=6'da işaret testi 6/6'da p=0.0156 verir; n=4'te en iyi 0.0625'ti, yani v3
tasarım gereği 0.05'i geçemiyordu.

## DURMA KURALI (ön-kayıtlı, bu koşudan önce yazıldı)

Bu satırlar §35'in sonucuna göre değil, **sonucundan önce** yazıldı.

- **Araç kapısı geçilmezse:** girişim hipotezi hakkında hiçbir şey yazılmaz.
  §26b'nin %33'ü tekrar üretilemiyorsa sorun taşıma değil, ve bu hat **kapanır**.
- **Kapı geçilir, SİNYAL çıkarsa:** tek izinli devam — ayrıştırma + güç
  (`nu4/additive` vs `nu2/delta`, 8-12 seed). Sonrasında yeni kol için **yeni
  karar** gerekir, otomatik devam yok.
- **Kapı geçilir, SONUÇSUZ/TERS çıkarsa:** hat kapanır. "Bir deney daha"
  yapılmaz. Kalan tek yol okuma/adresleme teşhisi ve o ayrı bir programdır.
- **Toplam bütçe tavanı:** bu hat için §35 v4 dahil **en fazla 2 koşu**. İkisi
  bittiğinde sonuç ne olursa olsun yayın hattına dönülür.

Gerekçe: bu projenin kaydı §30'dan §35'e altı bölüm ve hepsi negatif ya da
geçersiz; her biri "bir sonraki müdahale tutar" diye başladı. Durma kuralı
sonradan yazılamaz.

**Bu bir TARAMA deneyidir, doğrulayıcı değil.** "Kanıtlandı" denmeyecek.

In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, re, json, itertools
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
# [v4] AYRI klasor: v3 ayni TAG'leri kullaniyordu (carryv1_exp_s0...). Ayni
# klasore yazsaydik atlama mantigi v3 ciktilarini 'bitmis' sayip ESKI AYARLI
# checkpointleri kullanirdi. (DEVIR: 'farkli kollar asla ayni ada yazmamali'.)
CKDIR = os.path.join(BASE,'chain35v4'); os.makedirs(CKDIR, exist_ok=True)
try:
    from google.colab import drive; drive.mount('/content/drive')
    CKDIR = '/content/drive/MyDrive/hfp_chain35v4'; os.makedirs(CKDIR, exist_ok=True)
except Exception as e:
    print(f'Drive yok ({type(e).__name__}) -> yerel: {CKDIR}')
import torch
DEV_OK = torch.cuda.is_available()
print('repo:', REPO, '| ckpt:', CKDIR)
print('GPU:', torch.cuda.get_device_name(0) if DEV_OK else 'YOK (CPU)')
assert DEV_OK, 'GPU YOK. Accelerator > T4 sec. CPU\'da bu ayarlar saatler surer.'
print('  NOT: Bu notebooku AYNI ANDA iki kez calistirma — loglar cift basar,')
print('       iki surec ayni checkpointlere yazar ve her sey 2x yavaslar.')
for nu in (2,4): print(f'  dpfp_nu={nu} -> key_dim={2*64*nu}, M=({2*64*nu},64)={2*64*nu*64:,} float/katman')

In [ ]:
# --- 1b. ON-UCUS (preflight): ~1 dk. GERCEK KOSUYA BASLAMADAN ONCE.
# CI bu scripti kosmuyor. 2 adimlik sahte kosuyla (a) script ayakta mi,
# (b) [§35 v4] AYRILMIS metrik alanlari yaziliyor mu dogrulanir.
# AYRI ckpt klasoru -> gercek kosuyu kirletmez.
import tempfile, csv as _csv, math as _math
PRE = os.path.join(tempfile.gettempdir(), 'hfp_preflight35v4')
os.makedirs(PRE, exist_ok=True)
_penv = {**os.environ, 'PYTHONPATH': REPO, 'HFP_CKPT_DIR': PRE,
         'CC_STEPS':'2','CC_CTX':'64','CC_P':'2','CC_BS':'2','CC_CARRY_MAX':'2',
         'CC_DIST_EVERY':'32','CC_GAPS':'256','CC_TRIALS':'3',
         'CC_VAL_N':'16','CC_VAL_K':'2','CC_DPFP_NU':'2','CC_WRITE':'additive'}
_r = subprocess.run([sys.executable,'review_scripts/carry_curriculum.py','exp','0','300'],
                    cwd=REPO, env=_penv)
_f = os.path.join(PRE, 'carryv1_exp_s0_valloss.csv')
assert _r.returncode == 0, f'ON-UCUS BASARISIZ: cikis kodu {_r.returncode}. GERCEK KOSUYU BASLATMA.'
assert os.path.exists(_f), f'ON-UCUS BASARISIZ: CSV yazilmadi ({_f}). GERCEK KOSUYU BASLATMA.'
_row = next(_csv.DictReader(open(_f)))
_need = {'val_cross','val_cross_sem','val_inchunk','val_inchunk_sem','val_loss','val_k','val_n'}
assert _need <= set(_row), f'ON-UCUS BASARISIZ: eksik alan: {_need - set(_row)}'
_c, _i = float(_row['val_cross']), float(_row['val_inchunk'])
assert _math.isfinite(_c) and _math.isfinite(_i), f'ON-UCUS BASARISIZ: NaN/inf: {_row}'
# 2 adim egitilmis model ln(164)=5.10 civarinda olmali; 3.40'in cok altiysa
# maskeleme yanlis yerden okuyor demektir.
assert _c > 3.0, f'ON-UCUS SUPHELI: val_cross={_c:.3f}, 2 adimda bu kadar dusuk olamaz -> maske hatali olabilir'
print(f'\nON-UCUS TAMAM')
print(f'  val_cross   (cross-chunk, BIRINCIL) = {_c:.4f}')
print(f'  val_inchunk (chunk-ici, TESHIS)     = {_i:.4f}')
print(f'  referans: ln(164)=5.0999 hicbir sey ogrenilmemis | ln(30)=3.4012 deger tokenlari arasi sans')
print('  (Onemli olan degerler degil: alanlar ayrisiyor mu, dosya yaziliyor mu.)')

In [ ]:
# --- 2. EGITIM: 2 kol x 6 seed, §26b AYARLARI ---
import time
SEEDS = [0,1,2,3,4,5]
BASE_ARM  = (2,'additive','taban (nu2/additive)')
TREAT_ARM = (4,'delta',   'nu4 + delta')
ARMS = [BASE_ARM, TREAT_ARM]
# §26b/§27/§28a ile AYNI arac ayarlari. v3'un kisintisi araci bozmustu.
BASE_ENV = {**os.environ, 'PYTHONPATH': REPO, 'HFP_CKPT_DIR': CKDIR,
            'CC_CARRY_MAX':'16','CC_STEPS':'1200','CC_CTX':'256','CC_P':'6',
            'CC_DIST_EVERY':'64','CC_BS':'8','CC_GAPS':'256','CC_TRIALS':'30',
            'CC_VAL_N':'64','CC_VAL_K':'2'}
MODE = 'exp'                      # tek degisken kapasite/yazim olsun
def cfgtag(nu, wr):
    return '' if (nu==2 and wr=='additive') else f'_nu{nu}{wr[0]}'
_t_all = time.time()
for nu, wr, label in ARMS:
    for s in SEEDS:
        cfg = cfgtag(nu, wr)
        if os.path.exists(f'{CKDIR}/carryv1{cfg}_{MODE}_s{s}_valloss.csv'):
            print(f'[atla] {label} s{s}'); continue
        print(f'\n=== EGITIM {label} (nu={nu}, {wr}) s{s} ===', flush=True)
        _t0 = time.time()
        env = {**BASE_ENV, 'CC_DPFP_NU': str(nu), 'CC_WRITE': wr}
        subprocess.run([sys.executable,'review_scripts/carry_curriculum.py',MODE,str(s),'36000'],
                       cwd=REPO, env=env)
        _d = time.time()-_t0
        _kalan = (len(ARMS)*len(SEEDS)) - (ARMS.index((nu,wr,label))*len(SEEDS) + SEEDS.index(s) + 1)
        print(f'[sure] bu kol {_d/60:.1f} dk | kalan {_kalan} kol -> ~{_kalan*_d/3600:.1f} saat', flush=True)
print(f'\nEGITIM TAMAM — toplam {(time.time()-_t_all)/3600:.2f} saat')

In [ ]:
# --- 3. SONDA: eslesmis + eski, K in {0,1,2,4} ---
import csv
KS = [0,1,2,4]
MP_TRIALS = 100
for nu, wr, label in ARMS:
    for s in SEEDS:
        cfg = cfgtag(nu, wr)
        if not os.path.exists(f'{CKDIR}/carryv1{cfg}_{MODE}_s{s}.pt'):
            print(f'[eksik ckpt] {label} s{s}'); continue
        if os.path.exists(f'{CKDIR}/matchedv1{cfg}_{MODE}_s{s}.csv'):
            continue
        env = {**BASE_ENV, 'MP_DPFP_NU': str(nu), 'MP_WRITE': wr,
               'MP_KS': ','.join(map(str,KS)), 'MP_TRIALS': str(MP_TRIALS),
               'MP_CTX':'256', 'MP_P':'6', 'MP_DIST_EVERY':'64'}
        print(f'\n=== SONDA {label} s{s} ===', flush=True)
        subprocess.run([sys.executable,'review_scripts/matched_probe.py',MODE,str(s)],
                       cwd=REPO, env=env)
print('\nSONDA TAMAM')

In [ ]:
# --- 4. ARAC GECERLILIK KAPISI (once bu; gecmezse hukum YOK) ---
import csv, statistics as st
GATE = 25.0   # taban kol K=0 seed-ortalamasi, %. §26b: %33.2
def probe(nu, wr, field='matched_acc'):
    acc = {k: [] for k in KS}
    for s in SEEDS:
        f = f'{CKDIR}/matchedv1{cfgtag(nu,wr)}_{MODE}_s{s}.csv'
        if not os.path.exists(f): continue
        for r in csv.DictReader(open(f)):
            k = int(r['K'])
            if k in acc: acc[k].append(float(r[field]))
    return acc

_b = probe(*BASE_ARM[:2])
GATE_OK = bool(_b[0]) and st.mean(_b[0]) >= GATE
print('=== ARAC GECERLILIK KAPISI ===')
if _b[0]:
    print(f'  taban K=0 = {st.mean(_b[0]):.1f}%  [{min(_b[0]):.0f}-{max(_b[0]):.0f}]  (n={len(_b[0])} seed)')
    print(f'  esik {GATE:.0f}% | §26b referans %33.2 | v3 kosusu %8.5 (gecersizdi)')
else:
    print('  VERI YOK')
print(f'  -> KAPI {"GECILDI" if GATE_OK else "GECILMEDI"}')
if not GATE_OK:
    print('\n  KOSU GECERSIZ. Girisim hipotezi hakkinda HICBIR SEY yazilmaz.')
    print('  On-kayitli durma kurali: §26b\'nin %33\'u tekrar uretilemiyorsa sorun')
    print('  tasima degil, ve bu hat KAPANIR. Asagidaki hukum blogunu OKUMA.')

In [ ]:
# --- 5. BIRINCIL HUKUM: eslesmis cross-chunk kayip (yalniz hedef token) ---
import csv, math, statistics as st
def vloss(nu, wr, s, field='val_cross'):
    f = f'{CKDIR}/carryv1{cfgtag(nu,wr)}_{MODE}_s{s}_valloss.csv'
    if not os.path.exists(f): return None
    return float(next(csv.DictReader(open(f)))[field])

print('=== TESHIS: chunk-ici kayip (arac ogrendi mi?) ===')
for nu, wr, label in ARMS:
    xs = [vloss(nu,wr,s,'val_inchunk') for s in SEEDS]
    xs = [x for x in xs if x is not None]
    if xs: print(f'  {label:>22}: {st.mean(xs):.4f}  (sans ln(30)=3.4012)')

print('\n=== BIRINCIL: cross-chunk kayip (nat, K=2, n=64 ornek/seed) ===')
print(f"{'seed':>6} {'taban':>10} {'nu4+delta':>12} {'fark':>10}")
print('-'*42)
diffs = []
for s in SEEDS:
    b, t = vloss(*BASE_ARM[:2], s), vloss(*TREAT_ARM[:2], s)
    if b is None or t is None:
        print(f'{s:>6} {"—":>10} {"—":>12} {"eksik":>10}'); continue
    diffs.append(t-b)
    print(f'{s:>6} {b:>10.4f} {t:>12.4f} {t-b:>+10.4f}')
print('(ln(30)=3.4012 sans | ln(164)=5.0999 hicbir sey ogrenilmemis | DUSUK = iyi)')

print('\n=== ON-KAYITLI HUKUM (§35 v4 — TARAMA) ===')
if not GATE_OK:
    print('  ARAC KAPISI GECILMEDI -> HUKUM VERILMEZ. Kosu gecersiz.')
elif len(diffs) < len(SEEDS):
    print(f'  EKSIK VERI: {len(diffs)}/{len(SEEDS)} seed. Hukum verilmez.')
else:
    md = st.mean(diffs); sd = st.stdev(diffs); sem = sd/math.sqrt(len(diffs))
    nb = sum(1 for d in diffs if d < 0)
    print(f'  eslesmis ortalama D = {md:+.4f} nat  (SD {sd:.4f}, SEM {sem:.4f}, n={len(diffs)})')
    print(f'  isaret: {nb}/{len(diffs)} seed tedavi lehine | eslesmis t = {md/sem:+.2f}')
    print(f'  n={len(diffs)}: 6/6 isaret testi p=0.0156')
    if md <= -0.15 and nb == len(diffs):
        print('\n  => SINYAL VAR (tarama gecti, "kanitlandi" DEGIL).')
        print('     Tek izinli devam: ayristirma + guc (nu4/additive vs nu2/delta,')
        print('     8-12 seed). Sonrasi icin YENI KARAR gerekir, otomatik devam yok.')
    elif md >= 0.15 and nb == 0:
        print('\n  => TERS YON: girisim hipotezi bu yonde curudu. HAT KAPANIR.')
    else:
        print('\n  => SONUCSUZ (null DEGIL). On-kayitli durma kurali geregi:')
        print('     HAT KAPANIR, "bir deney daha" yapilmaz. Yayin hattina donulur.')

In [ ]:
# --- 6. IKINCIL (betimleyici): sonda K taramasi ---
import statistics as st
for field in ('matched_acc','matched_logp'):
    print(f'\n--- {field} --- ortalama [min-max] | n={len(SEEDS)} seed x {MP_TRIALS} deneme')
    print(f"{'kol':>22} " + ' '.join(f'K={k:<14}' for k in KS))
    for nu, wr, label in ARMS:
        a = probe(nu, wr, field); row = []
        for k in KS:
            row.append(f'{st.mean(a[k]):6.1f} [{min(a[k]):.0f}-{max(a[k]):.0f}]' if a[k] else '     —        ')
        print(f'{label:>22} ' + ' '.join(row))
print('\n(sans %3.3. BETIMLEYICI — hukum birincil metrikte verildi.')
print(' §26b bu sondada seed-basi %8.5-69.5 yayilim olctu; tek basina yorumlanmaz.)')